In [0]:
# ==========================================
# IMPORTS
# ==========================================

from pyspark.sql.functions import *
from pyspark.sql.types import *

from pyspark.sql.window import Window

import uuid
import time

In [0]:
# ==========================================
# VARIABLES GOLD
# ==========================================

catalog_name = "retail_catalog"

gold_schema = "gold"

gold_base_path = "abfss://gold@stretailmaxdev01.dfs.core.windows.net"

silver_base_path = "abfss://silver@stretailmaxdev01.dfs.core.windows.net"

In [0]:
# ==========================================
# CREAR SCHEMA GOLD
# ==========================================

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS
{catalog_name}.{gold_schema}
MANAGED LOCATION '{gold_base_path}'
""")

print("Schema GOLD creado")

Schema GOLD creado


In [0]:
# ==========================================
# CARGAR TABLAS SILVER
# ==========================================

clientes_df = spark.read.format("delta").load(
    f"{silver_base_path}/dim_clientes"
)

articulos_df = spark.read.format("delta").load(
    f"{silver_base_path}/dim_articulos"
)

proveedores_df = spark.read.format("delta").load(
    f"{silver_base_path}/dim_proveedores"
)

tiendas_df = spark.read.format("delta").load(
    f"{silver_base_path}/dim_tiendas"
)

ventas_df = spark.read.format("delta").load(
    f"{silver_base_path}/fact_ventas"
)

inventario_df = spark.read.format("delta").load(
    f"{silver_base_path}/fact_stock_diario"
)

devoluciones_df = spark.read.format("delta").load(
    f"{silver_base_path}/fact_devoluciones"
)

print("Tablas Silver cargadas")

Tablas Silver cargadas


In [0]:
# ==========================================
# DIM_CLIENTES
# ==========================================

dim_clientes = clientes_df \
    .withColumn(
        "antiguedad_dias",
        datediff(current_date(), to_date(col("fec_registro")))
    ) \
    .withColumn(
        "genero_std",
        when(col("genero") == "M", "M")
        .when(col("genero") == "F", "F")
        .otherwise("NO INFORMADO")
    ) \
    .fillna({
        "rango_edad": "26-35"
    }) \
    .withColumn(
        "cliente_activo_flag",
        when(col("activo") == True, 1).otherwise(0)
    )

gold_path = f"{gold_base_path}/dim_clientes"

dim_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{gold_schema}.dim_clientes
USING DELTA
LOCATION '{gold_path}'
""")

print("dim_clientes creada")

dim_clientes creada


In [0]:
# ==========================================
# DIM_TIENDAS
# ==========================================

dim_tiendas = tiendas_df \
    .withColumn(
        "tipo_tienda_std",
        when(
            upper(col("tipo_tienda")).like("%HIPER%"),
            "HIPERMERCADO"
        )
        .when(
            upper(col("tipo_tienda")).like("%SUPER%"),
            "SUPERMERCADO"
        )
        .otherwise("OTROS")
    ) \
    .withColumn(
        "zona_distribucion",
        when(
            upper(col("id_pais")) == "ECUADOR",
            "NORTE"
        )
        .when(
            upper(col("id_pais")) == "COLOMBIA",
            "SUR"
        )
        .otherwise("CENTRO")
    )

gold_path = f"{gold_base_path}/dim_tiendas"

dim_tiendas.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{gold_schema}.dim_tiendas
USING DELTA
LOCATION '{gold_path}'
""")

print("dim_tiendas creada")

dim_tiendas creada


In [0]:
# ==========================================
# DIM PRODUCTOS
# ==========================================

articulos_df = spark.read.format("delta").load(
    f"{silver_base_path}/dim_articulos"
)

proveedores_df = spark.read.format("delta").load(
    f"{silver_base_path}/dim_proveedores"
)

dim_productos = articulos_df.alias("a") \
    .join(
        proveedores_df.alias("p"),
        col("a.id_proveedor") == col("p.id_proveedor"),
        "left"
    ) \
    .select(
        col("a.art_id"),
        col("a.cod_barra"),
        col("a.desc_art"),

        col("a.id_categ_n1"),
        col("a.id_categ_n2"),
        col("a.id_categ_n3"),

        col("a.id_proveedor"),

        col("p.razon_social").alias("nombre_proveedor"),

        col("p.tiempo_repo_dias"),

        col("a.precio_lista"),

        col("a.peso_kg"),

        col("a.unid_medida"),

        col("a.activo"),

        (
            col("a.precio_lista") * 0.30
        ).alias("margen_estimado_categoria")
    )

gold_path = f"{gold_base_path}/dim_productos"

dim_productos.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{gold_schema}.dim_productos
USING DELTA
LOCATION '{gold_path}'
""")

print("dim_productos creada")

display(dim_productos.limit(10))

dim_productos creada


art_id,cod_barra,desc_art,id_categ_n1,id_categ_n2,id_categ_n3,id_proveedor,nombre_proveedor,tiempo_repo_dias,precio_lista,peso_kg,unid_medida,activo,margen_estimado_categoria
382,2154915200180,Tuna,Bebés,Ropa bebé,frente,334,Mar Recio Cortés S.L.N.E,20,902.79,4.04,LT,true,270.837
382,2154915200180,Tuna,Bebés,Ropa bebé,frente,334,Mar Recio Cortés S.L.N.E,20,902.79,4.04,LT,true,270.837
382,2154915200180,Tuna,Bebés,Ropa bebé,frente,334,Mar Recio Cortés S.L.N.E,20,902.79,4.04,LT,true,270.837
382,2154915200180,Tuna,Bebés,Ropa bebé,frente,334,Mar Recio Cortés S.L.N.E,20,902.79,4.04,LT,true,270.837
382,2154915200180,Tuna,Bebés,Ropa bebé,frente,334,Mar Recio Cortés S.L.N.E,20,902.79,4.04,LT,true,270.837
452,9676627676747,Steel Fish,Bebés,Ropa bebé,constitución,461,Familia Barceló S.Coop.,26,414.85,9.5,KG,true,124.455
452,9676627676747,Steel Fish,Bebés,Ropa bebé,constitución,461,Familia Barceló S.Coop.,26,414.85,9.5,KG,true,124.455
452,9676627676747,Steel Fish,Bebés,Ropa bebé,constitución,461,Familia Barceló S.Coop.,26,414.85,9.5,KG,true,124.455
452,9676627676747,Steel Fish,Bebés,Ropa bebé,constitución,461,Familia Barceló S.Coop.,26,414.85,9.5,KG,true,124.455
452,9676627676747,Steel Fish,Bebés,Ropa bebé,constitución,461,Familia Barceló S.Coop.,26,414.85,9.5,KG,true,124.455


In [0]:
# ==========================================
# FACT_VENTAS
# ==========================================

fact_ventas = ventas_df.alias("v") \
    .join(
        dim_clientes.alias("c"),
        col("v.id_miembro") == col("c.id_miembro"),
        "left"
    ) \
    .select(
        col("v.id_trans"),
        col("v.id_miembro"),
        col("v.id_tienda"),
        col("v.art_id"),
        col("v.fec_trans"),
        col("v.hra_trans"),
        col("v.qty_vendida"),
        col("v.precio_unitario_venta"),
        col("v.descuento_aplicado"),
        col("v.tipo_pago"),
        col("v.canal_venta"),
        col("v.ingestion_timestamp"),
        col("v.source_system"),
        col("v.batch_id"),
        col("v.year"),
        col("v.month"),
        col("v.day"),
        col("c.id_miembro").alias("cliente_validado")
    ) \
    .withColumn(
        "cliente_anonimo_flag",
        when(col("cliente_validado").isNull(), 1)
        .otherwise(0)
    ) \
    .withColumn(
        "vr_venta_neto",
        (
            col("qty_vendida") *
            col("precio_unitario_venta")
        ) - col("descuento_aplicado")
    ) \
    .withColumn(
        "venta_descuento_flag",
        when(col("descuento_aplicado") > 0, 1)
        .otherwise(0)
    ) \
    .withColumn(
        "fec_venta",
        to_date(col("fec_trans"))
    )

gold_path = f"{gold_base_path}/fact_ventas"

fact_ventas.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("fec_venta") \
    .save(gold_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{gold_schema}.fact_ventas
USING DELTA
LOCATION '{gold_path}'
""")

print("fact_ventas creada")

fact_ventas creada


In [0]:
# ==========================================
# FACT INVENTARIO
# ==========================================

from pyspark.sql.functions import *

inventario_df = spark.read.format("delta").load(
    f"{silver_base_path}/fact_stock_diario"
)

dim_productos = spark.read.format("delta").load(
    f"{gold_base_path}/dim_productos"
)

# ==========================================
# JOIN LIMPIO
# SOLO TRAER tiempo_repo_dias
# ==========================================

fact_inventario = inventario_df.alias("i") \
    .join(
        dim_productos.select(
            "art_id",
            "tiempo_repo_dias"
        ).dropDuplicates(["art_id"]).alias("p"),
        col("i.art_id") == col("p.art_id"),
        "left"
    ) \
    .select(
        "i.*",
        "p.tiempo_repo_dias"
    )

# ==========================================
# PROMEDIO CONSUMO 14 DIAS
# ==========================================

fact_inventario = fact_inventario.withColumn(
    "promedio_consumo_14dias",
    round(
        col("stock_fisico") / 14,
        2
    )
)

# ==========================================
# COBERTURA DIAS
# ==========================================

fact_inventario = fact_inventario.withColumn(
    "cobertura_dias",
    when(
        col("promedio_consumo_14dias") > 0,
        round(
            col("stock_fisico") /
            col("promedio_consumo_14dias"),
            2
        )
    ).otherwise(0)
)

# ==========================================
# ALERTA QUIEBRE
# ==========================================

fact_inventario = fact_inventario.withColumn(
    "alerta_quiebre",
    when(
        (col("cobertura_dias") < col("tiempo_repo_dias")) &
        (col("promedio_consumo_14dias") > 0),
        1
    ).otherwise(0)
)

# ==========================================
# DIFERENCIA VS STOCK MINIMO
# ==========================================

fact_inventario = fact_inventario.withColumn(
    "diferencia_stock_min",
    col("stock_fisico") - col("stock_minimo_config")
)

# ==========================================
# WRITE GOLD
# ==========================================

gold_path = f"{gold_base_path}/fact_inventario"

fact_inventario.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

# ==========================================
# CREATE TABLE GOLD
# ==========================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{gold_schema}.fact_inventario
USING DELTA
LOCATION '{gold_path}'
""")

print("fact_inventario creada")

display(fact_inventario.limit(10))

fact_inventario creada


id_snapshot,art_id,id_tienda,fec_snapshot,stock_fisico,stock_transito,stock_reservado,stock_minimo_config,stock_maximo_config,ingestion_timestamp,source_system,batch_id,year,month,day,tiempo_repo_dias,promedio_consumo_14dias,cobertura_dias,alerta_quiebre,diferencia_stock_min
118101,2594,53,2025-08-27,209,27,49,33,846,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,27,14.93,14.0,1,176
118275,1163,140,2025-08-30,45,73,4,26,877,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,28,3.21,14.02,1,19
118330,4222,145,2025-03-07,131,98,41,23,814,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,22,9.36,14.0,1,108
118332,2636,70,2025-11-20,113,29,33,47,616,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,17,8.07,14.0,1,66
118343,2832,133,2025-10-08,235,67,33,22,376,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,24,16.79,14.0,1,213
118956,1996,51,2025-11-21,82,31,43,45,670,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,3,5.86,13.99,0,37
119022,4846,148,2025-05-20,62,84,23,43,564,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,4,4.43,14.0,0,19
119187,4905,19,2025-08-14,30,63,16,13,488,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,6,2.14,14.02,0,17
119275,2965,57,2025-05-16,80,99,47,17,793,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,30,5.71,14.01,1,63
119807,1921,45,2025-09-04,103,78,11,39,508,2026-05-19T21:39:00.91667Z,Azure_SQL_Retailmax,98c6d6a7-58b9-4349-a49f-ab93fe7673d5,2026,05,19,11,7.36,13.99,0,64


In [0]:
# ==========================================
# FACT DEVOLUCIONES
# ==========================================

from pyspark.sql.functions import *

devoluciones_df = spark.read.format("delta").load(
    f"{silver_base_path}/fact_devoluciones"
)

ventas_df = spark.read.format("delta").load(
    f"{gold_base_path}/fact_ventas"
)

dim_productos = spark.read.format("delta").load(
    f"{gold_base_path}/dim_productos"
)

# ==========================================
# JOIN CONTROLADO
# ==========================================

fact_devoluciones = devoluciones_df.alias("d") \
    .join(
        ventas_df.select(
            "id_trans",
            "qty_vendida",
            "precio_unitario_venta",
            "canal_venta"
        ).alias("v"),
        col("d.id_trans_origen") == col("v.id_trans"),
        "left"
    ) \
    .join(
        dim_productos.select(
            "art_id",
            "id_categ_n1",
            "id_proveedor"
        ).dropDuplicates(["art_id"]).alias("p"),
        col("d.art_id") == col("p.art_id"),
        "left"
    ) \
    .select(
        col("d.id_devolucion"),
        col("d.id_trans_origen"),
        col("d.art_id"),
        col("d.id_tienda"),
        col("d.fec_devolucion"),

        col("d.qty_devuelta"),

        col("d.motivo_cod"),

        col("d.canal_devolucion"),

        col("d.estado_devolucion"),

        col("d.vr_reembolso"),

        col("p.id_categ_n1"),

        col("p.id_proveedor"),

        col("v.precio_unitario_venta"),

        col("v.qty_vendida")
    )

# ==========================================
# MOTIVO LEGIBLE
# ==========================================

fact_devoluciones = fact_devoluciones.withColumn(
    "motivo_desc",
    when(
        col("motivo_cod") == "DAM",
        "Producto Dañado"
    ).when(
        col("motivo_cod") == "ERR",
        "Error Cliente"
    ).otherwise("Otros")
)

# ==========================================
# PRECIO ORIGINAL
# ==========================================

fact_devoluciones = fact_devoluciones.withColumn(
    "precio_original",
    col("precio_unitario_venta")
)

# ==========================================
# TASA DEVOLUCION
# ==========================================

fact_devoluciones = fact_devoluciones.withColumn(
    "tasa_devolucion",
    when(
        col("qty_vendida") > 0,
        round(
            (
                col("qty_devuelta") /
                col("qty_vendida")
            ) * 100,
            2
        )
    ).otherwise(0)
)

# ==========================================
# WRITE GOLD
# ==========================================

gold_path = f"{gold_base_path}/fact_devoluciones"

fact_devoluciones.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

# ==========================================
# CREATE TABLE
# ==========================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{gold_schema}.fact_devoluciones
USING DELTA
LOCATION '{gold_path}'
""")

print("fact_devoluciones creada")

display(fact_devoluciones.limit(10))

fact_devoluciones creada


id_devolucion,id_trans_origen,art_id,id_tienda,fec_devolucion,qty_devuelta,motivo_cod,canal_devolucion,estado_devolucion,vr_reembolso,id_categ_n1,id_proveedor,precio_unitario_venta,qty_vendida,motivo_desc,precio_original,tasa_devolucion
234,754239,4160,128,2026-02-25,1,Producto incompleto,Mobile,Rechazada,139.49,Tecnología,228,139.49,3,Otros,139.49,33.33
364,182715,998,54,2025-09-17,3,Error talla,Web,Pendiente,478.53,Hogar,3,159.51,5,Otros,159.51,60.0
624,28686,4280,146,2025-12-28,1,Producto defectuoso,Tienda,Pendiente,154.06,Tecnología,337,154.06,4,Otros,154.06,25.0
841,130152,2609,47,2026-04-07,1,No era lo esperado,Web,Pendiente,264.95,Moda,799,264.95,1,Otros,264.95,100.0
848,435149,2945,66,2026-03-10,2,No era lo esperado,Mobile,Pendiente,659.6,Tecnología,211,329.8,2,Otros,329.8,100.0
1452,258102,163,102,2025-11-15,2,No era lo esperado,Tienda,Pendiente,793.04,Bebés,300,396.52,3,Otros,396.52,66.67
1630,475504,3600,7,2025-09-22,1,No era lo esperado,Web,Aprobada,122.13,Moda,235,122.13,2,Otros,122.13,50.0
1801,673219,1867,84,2026-03-25,1,Error talla,Mobile,Pendiente,57.26,Tecnología,61,57.26,1,Otros,57.26,100.0
2238,133850,1976,47,2026-02-21,2,No era lo esperado,Tienda,Aprobada,472.16,Hogar,278,236.08,2,Otros,236.08,100.0
2502,806891,887,8,2026-05-12,1,No era lo esperado,Mobile,Aprobada,517.3,Bebés,23,517.3,2,Otros,517.3,50.0


In [0]:
# ==========================================
# FACT RFM CLIENTES
# ==========================================

from pyspark.sql.window import Window
from pyspark.sql.functions import *

ventas_df = spark.read.format("delta").load(
    f"{gold_base_path}/fact_ventas"
)

# ==========================================
# CLIENTES ACTIVOS 180 DIAS
# ==========================================

clientes_activos = ventas_df \
    .groupBy("id_miembro") \
    .agg(
        max("fec_venta").alias("ultima_compra")
    ) \
    .filter(
        datediff(
            current_date(),
            col("ultima_compra")
        ) <= 180
    )

# ==========================================
# VENTAS ULTIMOS 90 DIAS
# ==========================================

ventas_90 = ventas_df.filter(
    col("fec_venta") >= date_sub(
        current_date(),
        90
    )
)

# ==========================================
# SOLO CLIENTES ACTIVOS
# ==========================================

ventas_90 = ventas_90.join(
    clientes_activos.select("id_miembro"),
    "id_miembro",
    "inner"
)

# ==========================================
# CALCULO RFM
# ==========================================

rfm_df = ventas_90 \
    .groupBy("id_miembro") \
    .agg(
        datediff(
            current_date(),
            max("fec_venta")
        ).alias("recency"),

        countDistinct("id_trans").alias("frequency"),

        round(
            sum("vr_venta_neto"),
            2
        ).alias("monetary")
    )

# ==========================================
# SCORING QUINTILES
# ==========================================

window_spec = Window.orderBy("recency")

rfm_df = rfm_df \
    .withColumn(
        "r_score",
        6 - ntile(5).over(
            Window.orderBy("recency")
        )
    ) \
    .withColumn(
        "f_score",
        ntile(5).over(
            Window.orderBy(col("frequency").desc())
        )
    ) \
    .withColumn(
        "m_score",
        ntile(5).over(
            Window.orderBy(col("monetary").desc())
        )
    )

# ==========================================
# SEGMENTO RFM
# ==========================================

rfm_df = rfm_df.withColumn(
    "segmento_rfm",
    concat(
        lit("R"),
        col("r_score"),
        lit("-F"),
        col("f_score"),
        lit("-M"),
        col("m_score")
    )
)

# ==========================================
# CLASIFICACION NEGOCIO
# ==========================================

rfm_df = rfm_df.withColumn(
    "grupo_valor",
    when(
        (col("r_score") >= 4) &
        (col("f_score") >= 4) &
        (col("m_score") >= 4),
        "Champions"
    ).when(
        col("r_score") >= 4,
        "Clientes Recientes"
    ).when(
        col("f_score") >= 4,
        "Frecuentes"
    ).otherwise("Estandar")
)

# ==========================================
# WRITE GOLD
# ==========================================

gold_path = f"{gold_base_path}/fact_rfm_clientes"

rfm_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{gold_schema}.fact_rfm_clientes
USING DELTA
LOCATION '{gold_path}'
""")

print("fact_rfm_clientes creada")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


fact_rfm_clientes creada


In [0]:
# ==========================================
# AGREGACION VENTAS DIARIAS
# ==========================================

from pyspark.sql.functions import *

fact_ventas_df = spark.read.format("delta").load(
    f"{gold_base_path}/fact_ventas"
)

dim_productos_df = spark.read.format("delta").load(
    f"{gold_base_path}/dim_productos"
)

# ==========================================
# JOIN PARA TRAER CATEGORIA
# ==========================================

ventas_enriquecidas = fact_ventas_df.alias("v") \
    .join(
        dim_productos_df.select(
            "art_id",
            "id_categ_n1"
        ).dropDuplicates(["art_id"]).alias("p"),
        col("v.art_id") == col("p.art_id"),
        "left"
    ) \
    .select(
        "v.*",
        "p.id_categ_n1"
    )

# ==========================================
# AGREGACION
# ==========================================

agg_ventas = ventas_enriquecidas \
    .groupBy(
        "fec_venta",
        "canal_venta",
        "id_tienda",
        "id_categ_n1"
    ) \
    .agg(
        round(
            sum("vr_venta_neto"),
            2
        ).alias("ventas_netas"),

        round(
            avg("vr_venta_neto"),
            2
        ).alias("ticket_promedio"),

        countDistinct("id_trans")
        .alias("transacciones")
    )

# ==========================================
# WRITE GOLD
# ==========================================

gold_path = f"{gold_base_path}/agg_ventas_diarias"

agg_ventas.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

# ==========================================
# CREATE TABLE
# ==========================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{gold_schema}.agg_ventas_diarias
USING DELTA
LOCATION '{gold_path}'
""")

print("agg_ventas_diarias creada")

display(agg_ventas.limit(10))

agg_ventas_diarias creada


fec_venta,canal_venta,id_tienda,id_categ_n1,ventas_netas,ticket_promedio,transacciones
2025-12-23,Mobile,31,Hogar,9566.56,2391.64,4
2025-12-08,Web,110,Moda,7193.3,1198.88,6
2025-12-24,Mobile,45,Bebés,6816.39,1363.28,5
2025-12-28,Tienda,4,Hogar,7072.95,1768.24,4
2025-12-04,Mobile,84,Hogar,13566.26,1695.78,8
2025-12-18,Tienda,121,Tecnología,1311.88,437.29,3
2025-11-12,Mobile,19,Bebés,2503.91,834.64,3
2025-11-20,Web,71,Tecnología,9083.74,2270.94,4
2025-12-10,Web,98,Tecnología,8974.71,2243.68,4
2025-12-12,Mobile,95,Moda,2184.92,2184.92,1


In [0]:
agg_devoluciones = fact_devoluciones \
    .groupBy(
        "motivo_desc",
        "id_categ_n1",
        "id_proveedor",
        "canal_devolucion"
    ) \
    .agg(
        round(avg("tasa_devolucion"),2)
        .alias("promedio_devolucion")
    )

agg_devoluciones.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base_path}/agg_devoluciones_categoria")

In [0]:
# ==========================================
# KPI EJECUTIVO DIARIO
# ==========================================

from pyspark.sql.functions import *

fact_ventas_df = spark.read.format("delta").load(
    f"{gold_base_path}/fact_ventas"
)

kpi_df = fact_ventas_df \
    .groupBy(
        "fec_venta",
        "canal_venta"
    ) \
    .agg(

        round(
            sum("vr_venta_neto"),
            2
        ).alias("ventas_netas"),

        round(
            avg("vr_venta_neto"),
            2
        ).alias("ticket_promedio"),

        countDistinct("id_trans")
        .alias("total_transacciones"),

        round(
            avg("venta_descuento_flag") * 100,
            2
        ).alias("porcentaje_ventas_descuento")
    )

# ==========================================
# WRITE GOLD
# ==========================================

gold_path = f"{gold_base_path}/kpi_ejecutivo_diario"

kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

# ==========================================
# CREATE TABLE
# ==========================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{gold_schema}.kpi_ejecutivo_diario
USING DELTA
LOCATION '{gold_path}'
""")

print("kpi_ejecutivo_diario creada")

display(kpi_df.limit(10))

kpi_ejecutivo_diario creada


fec_venta,canal_venta,ventas_netas,ticket_promedio,total_transacciones,porcentaje_ventas_descuento
2025-12-14,Mobile,2283665.57,1550.35,1473,24.24
2025-12-13,Mobile,2347890.54,1593.95,1473,26.48
2025-03-08,Tienda,1412632.27,1656.08,850,25.44
2025-04-27,Mobile,1258174.45,1570.75,801,25.47
2025-12-08,Web,2465242.79,1636.95,1506,25.03
2025-12-16,Mobile,2646866.0,1633.87,1620,24.38
2025-11-12,Web,2168277.07,1565.54,1385,23.83
2025-08-10,Tienda,1564314.28,1653.61,946,26.11
2025-07-08,Mobile,1414970.36,1467.81,964,25.93
2025-03-18,Web,1390129.83,1691.16,822,25.18


In [0]:
# ==========================================
# DATA QUALITY TESTS
# ==========================================

tests = []

# TEST 1
nulls = fact_ventas.filter(
    col("id_trans").isNull()
).count()

tests.append(
    ("fact_ventas_pk_not_null", nulls == 0)
)

# TEST 2
duplicates = fact_ventas.count() - \
             fact_ventas.dropDuplicates().count()

tests.append(
    ("fact_ventas_no_duplicates", duplicates == 0)
)

# TEST 3
negative_sales = fact_ventas.filter(
    col("vr_venta_neto") < 0
).count()

tests.append(
    ("ventas_no_negativas", negative_sales == 0)
)

# TEST 4
invalid_stock = fact_inventario.filter(
    col("stock_fisico") < 0
).count()

tests.append(
    ("stock_no_negativo", invalid_stock == 0)
)

# TEST 5
invalid_rfm = rfm_df.filter(
    col("grupo_valor").isNull()
).count()

tests.append(
    ("rfm_segmentado", invalid_rfm == 0)
)

tests_df = spark.createDataFrame(
    tests,
    ["test_name", "passed"]
)

display(tests_df)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


test_name,passed
fact_ventas_pk_not_null,true
fact_ventas_no_duplicates,false
ventas_no_negativas,true
stock_no_negativo,false
rfm_segmentado,true


In [0]:
%sql

OPTIMIZE retail_catalog.gold.fact_ventas
ZORDER BY (canal_venta, art_id, id_tienda);

OPTIMIZE retail_catalog.gold.fact_inventario
ZORDER BY (art_id, id_tienda);

OPTIMIZE retail_catalog.gold.fact_rfm_clientes
ZORDER BY (grupo_valor);

path,metrics
abfss://gold@stretailmaxdev01.dfs.core.windows.net/fact_rfm_clientes,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 54337), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1779288245630, 1779288247480, 4, 0, null, List(0, 0), null, 9, 9, 0, 0, null)"
